# Auction Price Prediction

**Tabular Regression Project** — Predict auction hammer price from item, auction, and bidder features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `hammer_price`

## 2 · Project Overview

We predict the **hammer price** (final auction sale price) of items from pre-auction estimates, bidder participation, item quality, and auction house characteristics. Price prediction at auctions is used by consignors, buyers, and auction houses for valuation and strategy.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given an item's pre-auction estimate range, number of bidders, lot position, category, auction house tier, provenance score, condition grade, online bidding availability, and preview duration, predict the **hammer price** at auction.

## 5 · Why This Project Matters

- **Consignors** use price predictions to choose when and where to sell.
- **Buyers** set bidding limits based on expected final prices.
- **Auction houses** use predictions for reserve price setting and lot ordering.
- Understanding **bidder competition** and **provenance** effects enables strategic consignment.
- A fascinating domain where **expertise meets data science**.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | 10 (estimate low/high, bidders, lot position, category, auction house tier, provenance, condition, online bidding, preview days) |
| **Target** | `hammer_price` (continuous, USD) |
| **Categorical** | category (6), auction_house_tier (3) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "hammer_price"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8000
estimate_low = np.random.lognormal(6, 1.5, n).clip(50, 500000)
estimate_high = estimate_low * np.random.uniform(1.2, 2.5, n)
num_bidders = np.random.randint(1, 30, n)
lot_position = np.random.randint(1, 200, n)
category = np.random.choice(["art", "jewelry", "antiques", "collectibles",
                              "wine", "watches"], n)
auction_house = np.random.choice(["top_tier", "mid_tier", "regional"], n, p=[0.2, 0.45, 0.35])
provenance_score = np.random.uniform(0, 10, n)
condition_grade = np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.15, 0.35, 0.30, 0.15])
online_bidding = np.random.choice([0, 1], n, p=[0.3, 0.7])
days_on_preview = np.random.randint(1, 30, n)

house_factor = np.where(auction_house == "top_tier", 1.4,
               np.where(auction_house == "mid_tier", 1.1, 1.0)).astype(float)

midpoint = (estimate_low + estimate_high) / 2
hammer = (midpoint * house_factor
          * (1 + num_bidders * 0.02)
          * (1 + provenance_score * 0.03)
          * (0.7 + condition_grade * 0.1)
          * (1 + online_bidding * 0.05)
          + np.random.normal(0, midpoint * 0.1, n))
hammer = np.maximum(hammer, estimate_low * 0.5)

df = pd.DataFrame({
    "estimate_low": estimate_low.astype(int),
    "estimate_high": estimate_high.astype(int),
    "num_bidders": num_bidders, "lot_position": lot_position,
    "category": category, "auction_house_tier": auction_house,
    "provenance_score": provenance_score,
    "condition_grade": condition_grade,
    "online_bidding_enabled": online_bidding,
    "days_on_preview": days_on_preview,
    "hammer_price": hammer.astype(int)
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["estimate_low", "estimate_high", "num_bidders",
                          "provenance_score", "condition_grade", "lot_position"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["hammer_price"], alpha=0.15, s=6)
    ax.set_xlabel(col); ax.set_ylabel("hammer_price"); ax.set_title(f"Price vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("category")["hammer_price"].median().sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Median Hammer Price by Category"); axes[0].set_xlabel("$")
df.groupby("auction_house_tier")["hammer_price"].median().sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Median Hammer Price by Auction House Tier"); axes[1].set_xlabel("$")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `hammer_price`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("Price ($)")
axes[1].hist(np.log1p(df[TARGET]), bins=40, color="darkorange", edgecolor="black", alpha=0.8)
axes[1].set_title(f"Log Distribution of {TARGET}"); axes[1].set_xlabel("log(Price)")
plt.tight_layout(); plt.show()
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Std: ${df[TARGET].std():,.0f}, Range: ${df[TARGET].min():,.0f} - ${df[TARGET].max():,.0f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["estimate_midpoint"] = (X_train["estimate_low"] + X_train["estimate_high"]) / 2
X_test["estimate_midpoint"] = (X_test["estimate_low"] + X_test["estimate_high"]) / 2

X_train["estimate_range"] = X_train["estimate_high"] - X_train["estimate_low"]
X_test["estimate_range"] = X_test["estimate_high"] - X_test["estimate_low"]

X_train["log_estimate_mid"] = np.log1p(X_train["estimate_midpoint"])
X_test["log_estimate_mid"] = np.log1p(X_test["estimate_midpoint"])

X_train["bidder_competition"] = X_train["num_bidders"] * X_train["online_bidding_enabled"]
X_test["bidder_competition"] = X_test["num_bidders"] * X_test["online_bidding_enabled"]

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Pre-auction estimates** (midpoint) are the strongest predictor — experts set estimates close to realized prices.
- **Number of bidders** drives prices above estimates — more competition creates bidding wars.
- **Auction house tier** multiplies prices — top-tier houses achieve 40% premiums via reputation and buyer network.
- **Provenance score** and **condition grade** act as quality multipliers.
- **Online bidding** widens the buyer pool, boosting prices by ~5%.

**Business takeaway:** Consign high-value items to top-tier houses when provenance is strong. Enable online bidding to maximize bidder competition. Items with high condition grades and provenance consistently beat estimates.

## 26 · Limitations

1. Synthetic data — real auction prices are influenced by art market trends, celebrity ownership, and sentiment.
2. No image data — visual appeal matters enormously at auction.
3. No time series — art market cycles affect all prices.
4. Condition grading is subjective in practice.
5. No buyer's premium included — total cost to buyer differs from hammer price.

## 27 · How to Improve This Project

1. Use real auction result data (Artnet, Christie's, Sotheby's APIs).
2. Add image features from lot photographs.
3. Include market trend indicators (art market index).
4. Model as multiplicative (log-linear) to better capture proportional effects.
5. Add artist/maker reputation scores for art/antiques.

## 28 · Production Considerations

- Provide pre-sale price estimates for marketing materials.
- Guide reserve price setting for consignors.
- Alert specialists when lots are predicted to significantly exceed estimates.
- Monitor market trends through rolling prediction accuracy.

## 29 · Common Mistakes

1. Not log-transforming the target — prices are right-skewed and multiplicative.
2. Ignoring the estimate range as a signal of valuation uncertainty.
3. Treating lot position as continuous without considering fatigue effects.
4. Not distinguishing between bought-in (unsold) and sold lots.
5. Using hammer price without adjusting for buyer's premium and taxes.

## 30 · Mini Challenge / Exercises

1. Predict log(hammer_price) instead — does it improve R²?
2. Calculate the "premium ratio" (hammer / estimate_mid) — which features predict it?
3. Remove estimates entirely — can the model predict price from other features alone?
4. Train only on art — does the model generalize to watches?
5. At what number of bidders does the price premium plateau?

## 31 · Final Summary / Key Takeaways

1. **Pre-auction estimates** are the strongest baseline — experts already embed domain knowledge.
2. **Bidder count** drives prices above estimates through competition.
3. **Auction house tier** commands significant price premiums.
4. **Provenance and condition** act as multiplicative quality factors.
5. **Log transformation** of price features is important for this skewed domain.
6. **Real deployment** needs image data, market trends, and artist/maker attributes.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))